In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
tf.get_logger().setLevel("ERROR")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/vadym-util/')
sys.path.append('/kaggle/input/cmi-competition-code')

import pandas as pd
import data_utils
import os
import utils
from sklearn.pipeline import Pipeline
from scipy.stats import randint
from skopt.space import Categorical, Integer
from sklearn_genetic.space import Categorical as ECat, Integer as EInt
from sklearn.model_selection import GridSearchCV
from sklearn_genetic import GASearchCV
from sklearn.model_selection import RandomizedSearchCV
from skopt import BayesSearchCV
from sklearn.model_selection import GroupKFold
from skopt.space import Categorical, Integer, Real
from sklearn_genetic.space import Categorical as ECat, Integer as EInt, Continuous as EFloat
from scipy.stats import randint, uniform, loguniform
from sklearn.metrics import f1_score, make_scorer
import numpy as np

from sklearn.model_selection import KFold
import importlib
warnings.filterwarnings('ignore', module='deap')

import utils
from sklearn.metrics import accuracy_score, classification_report

2026-05-03 16:42:04.713983: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777826524.940769      30 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777826525.009661      30 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777826525.506879      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777826525.506917      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777826525.506920      30 computation_placer.cc:177] computation placer alr

In [2]:
data_folder = data_utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
n_splits = 3
cv = GroupKFold(n_splits=n_splits)
scoring_metric = 'f1_macro'
model_target = 'gesture_action'

scoring = None

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
acc_cols = ['acc_x', 'acc_y', 'acc_z']

pipe_name = "temporal_extractor"
classifier_name = 'CNN_1D'

search_mode = "bayesian"  # grid, random, evolutionary, bayesian

pipe_name = 'temporal_extractor'
candidates = 30
generations = 2
tournament_size = 2
elitism = True
crossover_probability = 0.8
mutation_probability = 0.2

chosen_orientation = ['Seated Straight']

train_size = 0.7

In [4]:
train_df = raw_train_df.set_index('row_id')

if train_size is None:
    rows = (train_demo_df['adult_child'] == 1) & (train_demo_df['sex'] == 1) & (train_demo_df['handedness'] == 1)
    ideal_subject_ids = train_demo_df.loc[rows].sort_values(by='elbow_to_wrist_cm', ascending=False)['subject'].to_list()

    train_sample_df = train_df.loc[train_df['subject'].isin(ideal_subject_ids), :]
    train_sample_df = train_sample_df[train_sample_df['sequence_type'] == 'Target']
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    print(train_sample_df['sequence_id'].nunique())

elif train_size == 0:
    some_sequences = train_df['sequence_id'].unique()[10:]
    train_sample_df = train_df[train_df['sequence_id'].isin(some_sequences)]
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    
else:
    target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

    target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
    target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
    target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

    train_sample_df, test_sample_df = data_utils.sample_balanced_split(
        target_only_df,
        train_pct=train_size,
        test_pct=0.2
    )

if chosen_orientation is not None:
    train_sample_df = train_sample_df[train_sample_df['orientation'].isin(chosen_orientation)]

Train: 3224 seqs | 63.1%
Test:  632 seqs  | 12.4%


In [5]:
if search_mode == "bayesian":
    param_space = {
        # --- Preprocessing (locked from best run) ---
        f"{pipe_name}__acc_mode": Categorical(["velocity"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__use_highpass_fallback": Categorical([True]),
        f"{pipe_name}__fix_quaternion_sign": Categorical([True]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__include_mask": Categorical([False]),
        f"{pipe_name}__thm_mode": Categorical(["centered_diff"]),

        # --- Parameters to vary ---
        # Sampling rate: try lower (acts as smoothing) and original
        f"{pipe_name}__sampling_rate": Categorical([10, 15, 20, 25]),

        # Window size: scale with sampling rate
        # At 10Hz: 20-40 frames = 2-4 seconds
        # At 25Hz: 5-15 frames = 0.2-0.6 seconds
        f"{pipe_name}__window_size": Integer(5, 50),

        # Clip value: moderate clipping worked well
        f"{pipe_name}__clip_value": Categorical([20.0, 50.0, 100.0]),

        # Interpolation
        f"{pipe_name}__interp_mode": Categorical([None, "linear"]),

        # Smooth alpha: try a small amount of additional smoothing
        f"{pipe_name}__smooth_alpha": Categorical([None, 0.0, 0.1, 0.2]),

        # Rotation: rot6d worked, but try alternatives at low sampling rate
        f"{pipe_name}__rotation_mode": Categorical(["rot6d", "euler", "delta_euler", "quaternion"]),

        # TOF: pooled_diff worked, try variants
        f"{pipe_name}__tof_mode": Categorical(["pooled", "pooled_diff", "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["nan_interpolate", "far_500"]),

        # --- Classifier ---
        f"{classifier_name}__maxlen": Integer(50, 200),

        f"{classifier_name}__conv_filters": Categorical([
            "64-128-128",
            "64-128-256",
            "128-256-256",
            "128-256-512",
            "64-128-256-256",
        ]),

        f"{classifier_name}__kernel_sizes": Categorical([
            "3-3-3",
            "5-5-5",
            "3-5-5",
            "5-5-7",
            "3-5-7",
            "5-7-9",
        ]),

        f"{classifier_name}__pool_sizes": Categorical([
            "none-none-none",
            "2-2-2",
            "2-2-none",
            "2-none-none",
        ]),

        f"{classifier_name}__dense_units": Categorical([
            "none",
            "64",
            "128",
            "64-32",
            "128-64",
        ]),

        f"{classifier_name}__use_batch_norm": Categorical([True]),

        f"{classifier_name}__spatial_dropout": Real(0.0, 0.3),

        f"{classifier_name}__dropout": Real(0.1, 0.5),

        f"{classifier_name}__learning_rate": Real(5e-5, 5e-4, prior="log-uniform"),

        f"{classifier_name}__batch_size": Categorical([16, 32]),

        f"{classifier_name}__epochs": Categorical([120, 150, 200, 250]),

        f"{classifier_name}__patience": Categorical([15, 20, 25, 30]),
    }

elif search_mode == "evolutionary":
    param_space = {
        f"{pipe_name}__acc_mode": ECat(["raw", "smoothed", "velocity", "displacement", "jerk"]),
        f"{pipe_name}__linear_acc_mode": ECat([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__use_linear_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__sampling_rate": ECat([20, 25, 50]),
        f"{pipe_name}__compute_dt": ECat([True]),
        f"{pipe_name}__clip_value": ECat([None, 20.0, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": ECat([None, "linear"]),
        f"{pipe_name}__use_highpass_fallback": ECat([True]),
        f"{pipe_name}__window_size": EInt(3, 21),
        f"{pipe_name}__smooth_alpha": ECat([None, 0.2, 0.5, 0.8]),
        f"{pipe_name}__standardize": ECat([None, "mean_std"]),
        f"{pipe_name}__include_mask": ECat([False]),

        f"{pipe_name}__rotation_mode": ECat([None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": ECat([True, False]),

        f"{pipe_name}__tof_mode": ECat([None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": ECat(["nan_interpolate", "far_255", "far_500"]),
        f"{pipe_name}__thm_mode": ECat([None, "raw", "diff", "centered", "centered_diff"]),

        f"{classifier_name}__maxlen": EInt(16, 160),
        f"{classifier_name}__padding_value": ECat([-999.0]),
        f"{classifier_name}__conv_filters": ECat(["32", "64", "128", "32-64", "64-64", "64-128", "32-64-128", "64-128-128"]),
        f"{classifier_name}__kernel_sizes": ECat(["3", "5", "7", "3-3", "5-5", "3-5-7", "5-7-9"]),
        f"{classifier_name}__pool_sizes": ECat(["none", "2", "none-none", "2-none", "2-2", "none-none-none", "2-none-none", "2-2-none"]),
        f"{classifier_name}__dense_units": ECat(["none", "32", "64", "128", "64-32", "128-64"]),
        f"{classifier_name}__use_batch_norm": ECat([True, False]),
        f"{classifier_name}__spatial_dropout": EFloat(0.0, 0.3),
        f"{classifier_name}__dropout": EFloat(0.0, 0.5),
        f"{classifier_name}__learning_rate": EFloat(1e-4, 2e-3),
        f"{classifier_name}__batch_size": ECat([16, 32, 64]),
        f"{classifier_name}__epochs": ECat([60, 80, 120]),
        f"{classifier_name}__patience": ECat([8, 12, 20]),
    }

elif search_mode == "random":

    param_space = {
        f"{pipe_name}__acc_mode": ["raw", "smoothed", "velocity", "displacement", "jerk"],
        f"{pipe_name}__linear_acc_mode": [None, "baseline"],
        f"{pipe_name}__use_acc_magnitude": [False, True],
        f"{pipe_name}__use_linear_acc_magnitude": [False, True],
        f"{pipe_name}__sampling_rate": [20, 25, 50],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [None, 20.0, 50.0, 100.0],
        f"{pipe_name}__interp_mode": [None, "linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": randint(3, 22),
        f"{pipe_name}__smooth_alpha": [None, 0.2, 0.5, 0.8],
        f"{pipe_name}__standardize": [None, "mean_std"],
        f"{pipe_name}__include_mask": [False],

        f"{pipe_name}__rotation_mode": [None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"],
        f"{pipe_name}__fix_quaternion_sign": [False, True],

        f"{pipe_name}__tof_mode": [None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate", "far_255", "far_500"],
        f"{pipe_name}__thm_mode": [None, "raw", "diff", "centered", "centered_diff"],

        f"{classifier_name}__maxlen": randint(16, 161),
        f"{classifier_name}__padding_value": [-999.0],
        f"{classifier_name}__conv_filters": ["32", "64", "128", "32-64", "64-64", "64-128", "32-64-128", "64-128-128"],
        f"{classifier_name}__kernel_sizes": ["3", "5", "7", "3-3", "5-5", "3-5-7", "5-7-9"],
        f"{classifier_name}__pool_sizes": ["none", "2", "none-none", "2-none", "2-2", "none-none-none", "2-none-none", "2-2-none"],
        f"{classifier_name}__dense_units": ["none", "32", "64", "128", "64-32", "128-64"],
        f"{classifier_name}__use_batch_norm": [True, False],
        f"{classifier_name}__spatial_dropout": uniform(0.0, 0.3),
        f"{classifier_name}__dropout": uniform(0.0, 0.5),
        f"{classifier_name}__learning_rate": loguniform(1e-4, 2e-3),
        f"{classifier_name}__batch_size": [16, 32, 64],
        f"{classifier_name}__epochs": [60, 80, 120],
        f"{classifier_name}__patience": [8, 12, 20],
    }

elif search_mode == "grid":
    
    param_space = {
    # ===== FEATURE EXTRACTOR (temporal_extractor) =====
    f"{pipe_name}__acc_mode": ["velocity"],                    # Proven best
    f"{pipe_name}__linear_acc_mode": ["baseline"],             
    f"{pipe_name}__use_acc_magnitude": [True],                 
    f"{pipe_name}__use_linear_acc_magnitude": [True],          
    f"{pipe_name}__sampling_rate": [10],                       # Preserve temporal detail
    f"{pipe_name}__compute_dt": [True],                        
    f"{pipe_name}__clip_value": [50.0],                        
    f"{pipe_name}__interp_mode": ["linear"],                   
    f"{pipe_name}__use_highpass_fallback": [True],             
    f"{pipe_name}__window_size": [31],                         # Larger context (was 21)
    f"{pipe_name}__smooth_alpha": [0.0],                       # No smoothing preserves high-freq
    f"{pipe_name}__standardize": ["mean_std"],                 
    f"{pipe_name}__include_mask": [False],                     

    f"{pipe_name}__rotation_mode": ["rot6d"],                  # Most expressive (6D continuous)
    # Alternative: ["delta_euler"] also works well
    f"{pipe_name}__fix_quaternion_sign": [True],               

    f"{pipe_name}__tof_mode": ["pooled_diff"],                 
    f"{pipe_name}__tof_fill_mode": ["nan_interpolate"],        
    f"{pipe_name}__thm_mode": ["centered_diff"],               

    # ===== LARGER CNN CLASSIFIER =====
    f"{classifier_name}__maxlen": [100],                       # Longer sequences (was 200)
    f"{classifier_name}__padding_value": [-999.0],             

    # DEEPER + WIDER: Pyramid up to 256 filters
    f"{classifier_name}__conv_filters": ["128-256-256"],       # 3 conv layers, deeper
    f"{classifier_name}__kernel_sizes": ["5-5-5"],             # Larger kernels, consistent
    f"{classifier_name}__pool_sizes": ["2-2-2"],               # Progressive pooling
    
    # Keep minimal dense head (still "none" but with more conv features)
    f"{classifier_name}__dense_units": ["none"],               
    
    f"{classifier_name}__use_batch_norm": [True],              # Critical for deep models
    f"{classifier_name}__spatial_dropout": [0.2],              # Increased for more filters
    f"{classifier_name}__dropout": [0.4],                      # Higher regularization
    
    f"{classifier_name}__learning_rate": [1e-4],               # Lower LR for deeper net
    f"{classifier_name}__batch_size": [16],                    # Keep small for long sequences
    f"{classifier_name}__epochs": [200],                       # More epochs for convergence
    f"{classifier_name}__patience": [25],                      # Patient early stopping
}

In [6]:
importlib.reload(utils)         

pipeline = Pipeline([
    (pipe_name, utils.SequenceExtractor()),
    (classifier_name, utils.KerasCNN1DSequenceClassifier(
        target=model_target
    )),
])

if search_mode == "bayesian":
    search_obj = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "random":
    search_obj = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score = np.nan
    )

elif search_mode == "evolutionary":
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    search_obj = GASearchCV(
        estimator=pipeline,
        cv=cv,
        scoring=scoring,
        param_grid=param_space,
        population_size=candidates,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        criteria="max",
        n_jobs=1,
        verbose=True,
        keep_top_k=5,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "grid":
    search_obj = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

else:
    raise ValueError("search_mode must be one of: 'bayesian', 'random', 'evolutionary', 'grid'")

In [7]:
y = train_sample_df[['sequence_id', model_target]]
groups = train_sample_df['sequence_id']

print(f"--- {search_mode} Search ---")
if search_mode == 'evolutionary':
    search_obj.fit(train_sample_df, y)
else:
    search_obj.fit(train_sample_df, y, groups=groups)

--- bayesian Search ---
Fitting 3 folds for each of 1 candidates, totalling 3 fits


I0000 00:00:1777826646.941972      30 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777826646.947937      30 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1777826651.320593      73 service.cc:152] XLA service 0x7fb1fc00c970 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777826651.320646      73 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777826651.320668      73 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777826651.971798      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777826657.639678      73 device_compiler.h:188] Compiled clust

[CV 1/3] END CNN_1D__batch_size=16, CNN_1D__conv_filters=128-256-512, CNN_1D__dense_units=128-64, CNN_1D__dropout=0.2263198373948195, CNN_1D__epochs=200, CNN_1D__kernel_sizes=3-5-5, CNN_1D__learning_rate=0.00011217635889274475, CNN_1D__maxlen=161, CNN_1D__patience=20, CNN_1D__pool_sizes=2-2-none, CNN_1D__spatial_dropout=0.16498918254293524, CNN_1D__use_batch_norm=True, temporal_extractor__acc_mode=velocity, temporal_extractor__clip_value=50.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=False, temporal_extractor__interp_mode=linear, temporal_extractor__linear_acc_mode=baseline, temporal_extractor__rotation_mode=quaternion, temporal_extractor__sampling_rate=15, temporal_extractor__smooth_alpha=None, temporal_extractor__standardize=mean_std, temporal_extractor__thm_mode=centered_diff, temporal_extractor__tof_fill_mode=nan_interpolate, temporal_extractor__tof_mode=sensor_stats, temporal_extractor__use_acc_magnitude=Tr

2026-05-03 16:56:35.559167: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-03 16:56:35.629183: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng4{k11=2} for conv %cudnn-conv-bw-input.3 = (f32[5,256,1,81]{3,2,1,0}, u8[0]{0}) custom-call(f32[5,256,1,81]{3,2,1,0} %bitcast.14640, f32[256,256,1,5]{3,2,1,0} %bitcast.14644), window={size=1x5 pad=0_0x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_type="Conv2DBackpropInput" op_name="gradient_tape/functional_1/conv1d_3_1/convolution/Conv2DBackpropInput" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNon

[CV 1/3] END CNN_1D__batch_size=16, CNN_1D__conv_filters=64-128-256-256, CNN_1D__dense_units=none, CNN_1D__dropout=0.2733331207480901, CNN_1D__epochs=120, CNN_1D__kernel_sizes=3-5-5, CNN_1D__learning_rate=7.151844421050703e-05, CNN_1D__maxlen=163, CNN_1D__patience=25, CNN_1D__pool_sizes=2-none-none, CNN_1D__spatial_dropout=0.21104025933189122, CNN_1D__use_batch_norm=True, temporal_extractor__acc_mode=velocity, temporal_extractor__clip_value=50.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=False, temporal_extractor__interp_mode=None, temporal_extractor__linear_acc_mode=baseline, temporal_extractor__rotation_mode=quaternion, temporal_extractor__sampling_rate=10, temporal_extractor__smooth_alpha=0.0, temporal_extractor__standardize=mean_std, temporal_extractor__thm_mode=centered_diff, temporal_extractor__tof_fill_mode=far_500, temporal_extractor__tof_mode=pooled_diff, temporal_extractor__use_acc_magnitude=True, tempo

...

[CV 1/3] END CNN_1D__batch_size=32, CNN_1D__conv_filters=64-128-128, CNN_1D__dense_units=64-32, CNN_1D__dropout=0.39843654763573755, CNN_1D__epochs=200, CNN_1D__kernel_sizes=5-5-5, CNN_1D__learning_rate=0.0003467256142842097, CNN_1D__maxlen=200, CNN_1D__patience=25, CNN_1D__pool_sizes=none-none-none, CNN_1D__spatial_dropout=0.3, CNN_1D__use_batch_norm=True, temporal_extractor__acc_mode=velocity, temporal_extractor__clip_value=100.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=False, temporal_extractor__interp_mode=None, temporal_extractor__linear_acc_mode=baseline, temporal_extractor__rotation_mode=quaternion, temporal_extractor__sampling_rate=10, temporal_extractor__smooth_alpha=0.0, temporal_extractor__standardize=mean_std, temporal_extractor__thm_mode=centered_diff, temporal_extractor__tof_fill_mode=far_500, temporal_extractor__tof_mode=pooled, temporal_extractor__use_acc_magnitude=True, temporal_extractor__use_

2026-05-03 18:27:40.572421: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-03 18:27:40.828776: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


[CV 1/3] END CNN_1D__batch_size=32, CNN_1D__conv_filters=128-256-256, CNN_1D__dense_units=128, CNN_1D__dropout=0.20470323079280126, CNN_1D__epochs=150, CNN_1D__kernel_sizes=3-3-3, CNN_1D__learning_rate=5e-05, CNN_1D__maxlen=200, CNN_1D__patience=25, CNN_1D__pool_sizes=2-2-none, CNN_1D__spatial_dropout=0.3, CNN_1D__use_batch_norm=True, temporal_extractor__acc_mode=velocity, temporal_extractor__clip_value=20.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=False, temporal_extractor__interp_mode=linear, temporal_extractor__linear_acc_mode=baseline, temporal_extractor__rotation_mode=quaternion, temporal_extractor__sampling_rate=15, temporal_extractor__smooth_alpha=0.0, temporal_extractor__standardize=mean_std, temporal_extractor__thm_mode=centered_diff, temporal_extractor__tof_fill_mode=nan_interpolate, temporal_extractor__tof_mode=pooled, temporal_extractor__use_acc_magnitude=True, temporal_extractor__use_highpass_fallb

2026-05-03 18:38:26.656997: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-03 18:38:26.963619: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-03 18:38:31.359476: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-03 18:38:31.621937: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-03 18:38:45.929968: E external/local_xla/xla/stream_

[CV 1/3] END CNN_1D__batch_size=16, CNN_1D__conv_filters=128-256-256, CNN_1D__dense_units=128, CNN_1D__dropout=0.4030597637884866, CNN_1D__epochs=200, CNN_1D__kernel_sizes=3-5-7, CNN_1D__learning_rate=0.00012030435506926535, CNN_1D__maxlen=102, CNN_1D__patience=15, CNN_1D__pool_sizes=none-none-none, CNN_1D__spatial_dropout=0.15066952355277954, CNN_1D__use_batch_norm=True, temporal_extractor__acc_mode=velocity, temporal_extractor__clip_value=20.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=False, temporal_extractor__interp_mode=None, temporal_extractor__linear_acc_mode=baseline, temporal_extractor__rotation_mode=quaternion, temporal_extractor__sampling_rate=20, temporal_extractor__smooth_alpha=0.0, temporal_extractor__standardize=mean_std, temporal_extractor__thm_mode=centered_diff, temporal_extractor__tof_fill_mode=nan_interpolate, temporal_extractor__tof_mode=sensor_stats, temporal_extractor__use_acc_magnitude=Tr

In [8]:
# --- 1. Model Prediction & Evaluation ---
best_model = search_obj.best_estimator_
X_test = test_sample_df.copy()

# Get unique ground truth labels per sequence
y_true_seq = (test_sample_df[['sequence_id', model_target]]
              .drop_duplicates('sequence_id')
              .reset_index(drop=True))

y_pred_seq = best_model.predict(X_test)

# Calculate Accuracy
test_accuracy = accuracy_score(y_true_seq[model_target], y_pred_seq)

print(f"--- Final Test Results ---")
print(f"Best CV Score: {search_obj.best_score_:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true_seq[model_target], y_pred_seq))

# --- 2. Cleanly Append Test Results to CV Results ---
if hasattr(search_obj, 'cv_results_'):
    # Convert search results to DataFrame
    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    cv_results_df['search_mode'] = search_mode
    cv_results_df['target'] = model_target
    
    # Create a "Final Test" row matching the CV columns
    # We use 'params' to label it and put the accuracy in 'mean_test_score'
    test_result_row = pd.DataFrame({
        'params': ['FINAL_HOLD_OUT_TEST'],
        'mean_test_score': [test_accuracy],
        'std_test_score': [0],
        'rank_test_score': [0]
    })
    
    # Concat results - holes in the table (like split scores) fill with NaN
    final_report_df = pd.concat([cv_results_df, test_result_row], ignore_index=True)
    
    # Save the consolidated report
    file_path = f"{model_run_folder_name}{search_mode}_{classifier_name}_results.csv"
    final_report_df.to_csv(file_path, index=False)
    
    print(f"Results consolidated and saved to: {file_path}")

--- Final Test Results ---
Best CV Score: 0.6640
Test Accuracy: 0.5253

Classification Report:
                precision    recall  f1-score   support

   pinch skin       0.52      0.47      0.50       158
    pull hair       0.64      0.57      0.60       237
pull hairline       0.66      0.34      0.45        79
      scratch       0.41      0.61      0.49       158

     accuracy                           0.53       632
    macro avg       0.56      0.50      0.51       632
 weighted avg       0.55      0.53      0.53       632

Results consolidated and saved to: model_runs/bayesian_CNN_1D_results.csv
